# PCB part detector: tiled retrain on a free Kaggle GPU (Track A)

**Before running:**
1. *Add Data* → upload `pcb_tiles.zip` as a new dataset (made by `detector/prepare_tiles.py`).
2. *Settings* → Accelerator **GPU T4 x2** (or P100), Internet **on**.
3. *Run All*. Takes about 1–1.5 h. When it finishes, download `/kaggle/working/detector_export.zip`
   from the *Output* panel and give it back to Claude (it holds `parts.onnx`, `parts.json` and the training curves).

Model: YOLOv8**s** (about 3× the capacity of the current YOLOv8n), 1024 px tiles cut from photos at 2048 px.
The 22 test boards are never used for training or model selection.

In [ ]:
!pip -q install ultralytics==8.3.40 onnx onnxslim
import glob, json, os, shutil, time, yaml
from pathlib import Path
import torch
print(torch.__version__, torch.cuda.is_available(), torch.cuda.device_count())

In [ ]:
# find the uploaded dataset (folder that holds data.yaml + summary.json)
cands = [Path(p).parent for p in glob.glob('/kaggle/input/**/summary.json', recursive=True)]
assert cands, 'Upload pcb_tiles.zip as a Kaggle dataset first'
DATA = cands[0]
print(DATA, os.listdir(DATA))
summary = json.loads((DATA / 'summary.json').read_text())
print(json.dumps(summary, indent=1)[:800])
cfg = yaml.safe_load((DATA / 'data.yaml').read_text())
cfg['path'] = str(DATA)
Path('/kaggle/working/data.yaml').write_text(yaml.safe_dump(cfg, sort_keys=False))
names = [cfg['names'][i] for i in sorted(cfg['names'])]
names

In [ ]:
from ultralytics import YOLO
IMGSZ = summary['tile']                     # 1024
t0 = time.time()
model = YOLO('yolov8s.pt')                  # COCO-pretrained start point
model.train(data='/kaggle/working/data.yaml', epochs=100, imgsz=IMGSZ, batch=16, device=0, workers=4,
            patience=25, time=2.5, plots=True, project='/kaggle/working/runs', name='parts', exist_ok=True,
            seed=0, deterministic=True, fliplr=0.5, flipud=0.5, degrees=5, scale=0.3, close_mosaic=10,
            max_det=1000, cache='ram')
train_minutes = (time.time() - t0) / 60
print(f'{train_minutes:.0f} min')

In [ ]:
best = YOLO('/kaggle/working/runs/parts/weights/best.pt')
m = best.val(data='/kaggle/working/data.yaml', split='test', imgsz=IMGSZ, batch=16, device=0,
             project='/kaggle/working/runs', name='test', exist_ok=True, max_det=1000)
per_class = {names[c]: round(float(m.box.ap50[i]), 3) for i, c in enumerate(m.box.ap_class_index)}
print(round(float(m.box.map50), 3), per_class)

In [ ]:
onnx = Path(best.export(format='onnx', imgsz=IMGSZ, opset=12, simplify=True, dynamic=False))
b = summary['boards']
meta = {
    'names': names, 'imgsz': IMGSZ,
    'metrics': {'map50': round(float(m.box.map50), 3), 'map50_95': round(float(m.box.map), 3),
                'precision': round(float(m.box.mp), 3), 'recall': round(float(m.box.mr), 3)},
    'per_class_ap50': per_class,
    'training': {'base': 'yolov8s.pt', 'epochs_requested': 100, 'imgsz': IMGSZ,
                 'minutes': round(train_minutes, 1), 'smoke_test': False, 'device': 'kaggle-gpu'},
    'dataset': (f"Trained on {b['train']} boards, validated on {b['valid']}, tested on {b['test']} unseen boards "
                f"(as {summary['tiles']['train']} training tiles). Source: {summary['source']}."),
    'tiling': {'long_side': summary['long_side'], 'tile': summary['tile'], 'overlap': summary['overlap'],
               'metrics_are': 'per tile (see tiling_eval.json for whole boards)'},
}
out = Path('/kaggle/working/detector_export'); out.mkdir(exist_ok=True)
shutil.copy(onnx, out / 'parts.onnx')
shutil.copy('/kaggle/working/runs/parts/weights/best.pt', out / 'best.pt')
(out / 'parts.json').write_text(json.dumps(meta, indent=2))
for f in ['results.csv', 'results.png', 'confusion_matrix_normalized.png', 'PR_curve.png']:
    p = Path('/kaggle/working/runs/parts') / f
    if p.exists(): shutil.copy(p, out / f)
shutil.make_archive('/kaggle/working/detector_export', 'zip', out)
print(json.dumps(meta, indent=2))